## A first, dummy implementation of a GPT-2-like model

This notebook adapts the dummy GPT example from
[LLMs-from-scratch, Chapter 4](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch04/01_main-chapter-code/ch04.ipynb).
Copyright 2023–2026 Sebastian Raschka. Licensed under the Apache License, Version 2.0
(see `../LICENSES/Apache-2.0-LLMs-from-scratch.txt`). Modified for this repository:
parameter-counting examples and explanations have been added.

In [1]:
import torch
import torch.nn as nn

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,  # Vocabulary size of the GPT-2 BPE tokenizer
    "context_length": 1024,  # Maximum number of input tokens
    "emb_dim": 768,  # Embedding dimension
    "n_heads": 12,  # Number of attention heads
    "n_layers": 12,  # Number of transformer blocks
    "drop_rate": 0.1,  # Dropout rate
    "qkv_bias": False,  # Bias in the query, key, and value projections
}

### The model skeleton

The dummy model already has the full GPT data flow, from token IDs to logits:

1. token embeddings plus learned position embeddings;
2. dropout on the embeddings;
3. a stack of `n_layers` transformer blocks;
4. a final layer normalization;
5. a linear output head, with one score per vocabulary token.

The transformer block and the layer normalization are **placeholders** that
return their input unchanged. They keep the interface of the real classes, which
replace them later, so the model can already run end to end.

In [3]:
class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # A placeholder: the real block, with multi-head attention and a
        # feed-forward network, replaces it later.

    def forward(self, x):
        # This block does nothing and just returns its input.
        return x


class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        # The arguments only mimic the nn.LayerNorm interface: this placeholder
        # has no parameters.

    def forward(self, x):
        # This layer does nothing and just returns its input.
        return x

In [4]:
class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Use a placeholder for TransformerBlock
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        # Use a placeholder for LayerNorm
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        """Map token IDs to next-token logits.

        Args:
            in_idx: Token IDs of shape (batch_size, seq_len), with
                seq_len <= context_length.

        Returns:
            Logits of shape (batch_size, seq_len, vocab_size).
        """
        batch_size, seq_len = in_idx.shape

        # Transform token IDs into embeddings, and add a learned embedding for
        # each position 0 .. seq_len - 1. Broadcasting adds the same position
        # embeddings, (seq_len, emb_dim), to every sequence in the batch.
        tok_embeds = self.tok_emb(in_idx)  # (batch_size, seq_len, emb_dim)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds

        # Apply dropout to the embeddings; it is active only in training mode.
        x = self.drop_emb(x)

        # The stack of transformer blocks; each block keeps the shape
        # (batch_size, seq_len, emb_dim).
        x = self.trf_blocks(x)

        # Final layer normalization, before the output head.
        x = self.final_norm(x)

        # The output head maps each token vector to vocab_size scores (logits).
        # Logits are not probabilities: softmax turns the logits at a position
        # into the distribution of the next token.
        logits = self.out_head(x)  # (batch_size, seq_len, vocab_size)
        return logits

## Count existing parameters and estimate the complete architecture

The dummy Transformer blocks and LayerNorm have **no parameters**: their forward
methods only return their inputs. Counting this dummy model therefore measures
its embeddings and output head, not the complete GPT suggested by its config.

We use two reusable functions from `build_llm.model_parameters`:

- `count_parameters(model)` counts the scalar parameters actually registered in
  **any initialized PyTorch module**, regardless of architecture or device.
  Shared Parameter objects are counted once; buffers and activations are excluded.
  Set `trainable_only=True` to exclude frozen parameters.
- `estimate_gpt_parameters(config)` estimates a **standard full GPT architecture**
  from dimensions alone, without constructing it. The returned dictionary breaks
  the total down by component. This is an architectural estimate, not a count of
  the dummy blocks.

Install this repository once in editable mode, following `../QUICKSTART.md`.
In a terminal, activate `build-llm` and run `python -m pip install --no-deps -e .`
from the repository root. Then select a kernel from that environment and restart
it if it was already running. The import below will work even when the kernel's
working directory is `chap3`, without changing directories or `sys.path`.

In [5]:
from build_llm.model_parameters import count_parameters, estimate_gpt_parameters

# The meta device stores tensor shapes without allocating the large weight arrays.
# This instance is only for counting, not for running a numerical forward pass.
with torch.device("meta"):
    dummy_model = DummyGPTModel(GPT_CONFIG_124M)

print(
    f"Parameters currently present in DummyGPTModel: {count_parameters(dummy_model):,}"
)
print(f"Trainable parameters: {count_parameters(dummy_model, trainable_only=True):,}")

full_model_counts = estimate_gpt_parameters(GPT_CONFIG_124M)
print()
print("Estimated full GPT parameters (separate input and output weights):")
for component, parameter_count in full_model_counts.items():
    print(f"{component:>20}: {parameter_count:>15,}")

Parameters currently present in DummyGPTModel: 77,981,184
Trainable parameters: 77,981,184

Estimated full GPT parameters (separate input and output weights):
    token_embeddings:      38,597,376
 position_embeddings:         786,432
           attention:      28,320,768
        feed_forward:      56,669,184
         block_norms:          36,864
          final_norm:           1,536
         output_head:      38,597,376
               total:     163,009,536


### What the estimate assumes

Let $V$ be vocabulary size, $C$ context length, $D$ embedding width, $L$ the
number of blocks, and $H$ feed-forward hidden width (default $4D$).

| Component | Weight count, excluding biases |
| --- | --- |
| Token embeddings | $VD$ |
| Learned absolute position embeddings | $CD$ |
| Q, K, V and attention output projections, per block | $4D^2$ |
| Two feed-forward projections, per block | $2DH$ |
| Output head, if not tied to token embeddings | $VD$ |

The estimate adds two affine LayerNorms per block and one final LayerNorm,
including their scales and, by default, biases. It follows `qkv_bias` from the
config. Attention output and MLP biases default to enabled; the output head bias
defaults to disabled, matching the dummy head. Dropout, activations, masks, and
residual additions contribute no parameters. At fixed `emb_dim`, changing
`n_heads` only splits the same projections into heads and does not change the
standard multi-head attention parameter count.

The dictionary name `GPT_CONFIG_124M` is a label, not a guarantee of the count.
With separate input and output weights and `qkv_bias=False`, the full estimate is
**163,009,536** parameters. Tying the output weight to the token embedding removes
one $VD$ matrix. The example below also enables Q/K/V biases, producing the
familiar **124,439,808** count for the standard GPT-2 small structure.

Optional config fields allow `ff_hidden_dim`, `tie_embeddings`, `qkv_bias`,
`attention_output_bias`, `mlp_bias`, `layer_norm_bias`, and `output_bias` to vary.
These options affect only the estimate; they do not change `DummyGPTModel`.
The formula is not universal: gated MLPs, grouped-query attention, RMSNorm,
alternative position encodings, and shared Transformer blocks need different
formulas. Once a real implementation exists, use `count_parameters` on it as the
source of truth, optionally constructing it on the meta device to save memory.

In [6]:
# A new dictionary preserves the original notebook configuration.
tied_config = {**GPT_CONFIG_124M, "tie_embeddings": True, "qkv_bias": True}
tied_counts = estimate_gpt_parameters(tied_config)
print(
    f"Estimated full GPT with tied embeddings and Q/K/V biases: {tied_counts['total']:,}"
)

Estimated full GPT with tied embeddings and Q/K/V biases: 124,439,808
